In [ ]:
import pandas as pd
import numpy as np

# --- config ---
input_path = "./data/clean/CustomDataset/CustomDataset.inter"   # your raw csv file
output_path = "./data/clean/CustomDataset/CustomDataset_split.inter"

random_seed = 42
np.random.seed(random_seed)

# --- load ---
df = pd.read_csv(input_path, sep="\t")

#df.head()

# ensure correct column names for RecBole
#df.columns = ['user_id:token', 'item_id:token', 'rating:float']

# --- remove users who interacted with all items ---
n_items = df['item_id:token'].nunique()
user_item_counts = df.groupby('user_id:token')['item_id:token'].nunique()

bad_users = user_item_counts[user_item_counts >= n_items].index.tolist()
print(f"Found {len(bad_users)} users who interacted with all {n_items} items → removing them.")

df = df[~df['user_id:token'].isin(bad_users)]

# --- add split column ---
df['split:token'] = 'train'  # default

def assign_splits(group):
    """Assign 1 val, 1 test, rest train for each user."""
    if len(group) >= 2:
        chosen = np.random.choice(group.index, size=min(2, len(group)), replace=False)
        group.loc[chosen[0], 'split:token'] = 'val'
        if len(chosen) > 1:
            group.loc[chosen[1], 'split:token'] = 'test'
    return group

df = df.groupby('user_id:token', group_keys=False).apply(assign_splits)

# --- save ---
df.to_csv(output_path, sep='\t', index=False)

print(f"Saved dataset with splits → {output_path}")